## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 17 samples and 13911 columns.
First few rows of the data:


,months to death or last fu,status (alive - 0; dead/hospice - 1),sample_title,tumor type,"timing of immunotherapy use relative to rt (1 - after only, 2 - before and after)",OR4F29,OR4F16,NOC2L,KLHL17,PLEKHN1,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,22.566667,0,"SM-HCP (YY, YZ, Z1)",Metastatic Melanoma,1,1.419764,1.791127,3.737855,1.903591,2.064060,...,2.019139,0.822458,2.145742,1.870827,1.116687,0.611674,2.993061,2.801203,1.652033,2.512309
1,2.800000,1,SM-HCP (ZA-ZD),Metastatic Melanoma,2,1.018639,1.466905,3.547749,2.360976,0.528370,...,2.406441,1.182506,2.290120,2.175077,1.330957,1.099105,3.469617,3.294399,2.272340,2.570081
2,24.233333,1,SM-HCQ6 (S-V),Metastatic Melanoma,1,1.729009,1.896278,3.376944,1.931879,1.375554,...,1.673971,0.317861,1.191918,1.429004,0.612444,0.245000,2.238251,1.709542,0.729611,1.603659
3,16.033333,1,SM-HCP (WL-WP),Metastatic Melanoma,2,0.640884,1.105541,4.492214,3.184389,1.651799,...,3.288294,1.306839,2.585870,2.523845,1.427557,1.513530,3.765575,3.539695,2.527836,2.943814
4,51.966667,0,SM-HCQ7 (D-L),Metastatic Melanoma,1,0.782545,1.759684,4.663638,2.816783,0.801410,...,2.272631,1.077299,2.308460,2.089459,0.459387,1.406616,2.741814,2.622187,1.621544,2.399429


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
